# MLLM Teacher: Qwen2.5-Omni-3B on FSC

**Goal**: Load FSC, explore the task, then run frozen Qwen2.5-Omni-3B (4-bit quantised for local 3060) and verify hidden state extraction.

## 1. Imports & Device Check

In [1]:
import os
import sys

# Register FFmpeg shared DLLs so torchcodec can find avcodec/avformat/etc.
# The return value MUST be stored — if GC'd, the dir is removed from DLL search path.
if sys.platform == 'win32':
    _ffmpeg_dll_dir = None
    for _p in os.environ.get('PATH', '').split(';'):
        if _p and os.path.exists(os.path.join(_p, 'avcodec-62.dll')):
            _ffmpeg_dll_dir = os.add_dll_directory(_p)
            break
    if _ffmpeg_dll_dir is None:
        print("WARNING: avcodec-62.dll not found on PATH — torchcodec will fail")
    else:
        print(f"FFmpeg DLL dir registered: {_p}")

import torch
import numpy as np
from pathlib import Path
from IPython.display import Audio as IPyAudio, display  # aliased — Cell 2 imports datasets.Audio
                                                         # which would overwrite plain 'Audio'

print(f"PyTorch : {torch.__version__}")

FFmpeg DLL dir registered: C:\Users\lenovo\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg.Shared_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.1-full_build-shared\bin
PyTorch : 2.9.1+cu126


## 2. Load FSC Dataset

In [2]:
from datasets import load_dataset, Audio

CACHE_DIR = Path(r"D:\msc_AI\individual_project\multimodal-distillation-for-extreme-edge\data")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

fsc = load_dataset("s3prl/superb", name="ic", cache_dir=str(CACHE_DIR))
print(fsc)

DatasetDict({
    train: Dataset({
        features: ['file', 'audio', 'speaker_id', 'text', 'action', 'object', 'location'],
        num_rows: 23132
    })
    validation: Dataset({
        features: ['file', 'audio', 'speaker_id', 'text', 'action', 'object', 'location'],
        num_rows: 3118
    })
    test: Dataset({
        features: ['file', 'audio', 'speaker_id', 'text', 'action', 'object', 'location'],
        num_rows: 3793
    })
})


In [3]:
# Device check — deferred until datasets/pyarrow are already in sys.modules
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA    : True
GPU     : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM    : 6.4 GB


In [4]:
# Decode integer slot labels -> strings
features       = fsc["train"].features
action_names   = features["action"].names
object_names   = features["object"].names
location_names = features["location"].names

def decode_intent(example):
    a = action_names[example["action"]]
    o = object_names[example["object"]]
    l = location_names[example["location"]]
    example["intent"] = f"{a}_{o}_{l}"
    return example

fsc = fsc.map(decode_intent)

intent_labels = sorted(set(fsc["train"]["intent"]))
label2id = {l: i for i, l in enumerate(intent_labels)}
id2label  = {i: l for l, i in label2id.items()}
print(f"Train: {len(fsc['train']):,}  |  Val: {len(fsc['validation']):,}  |  Test: {len(fsc['test']):,}")
print(f"Unique intents: {len(intent_labels)}")

Train: 23,132  |  Val: 3,118  |  Test: 3,793
Unique intents: 31


In [5]:
import io
import soundfile as sf

# datasets 4.8.5 removed the soundfile fallback and requires torchcodec for Audio
# decoding — but torchcodec 0.12.0 links against a different FFmpeg soversion than
# Gyan's FFmpeg 8.1.1 (avutil-60 vs what torchcodec expects).  Using decode=False
# tells datasets to return raw bytes without calling torchcodec at all; we decode
# manually with soundfile, which produces an identical float32 array.
fsc = fsc.cast_column("audio", Audio(sampling_rate=16000, decode=False))

## 3. Listen to Examples

Play a few clips with different intents to get a feel for the task.

FSC is a **spoken smart-home command** dataset. Each utterance is tagged with three slots:
- **action**: what to do (activate / deactivate / increase / decrease / change language / bring)
- **object**: what to do it to (lights / music / heat / lamp / newspaper / juice / socks / …)
- **location**: where (none / bedroom / kitchen / washroom)

Combined intent = `action_object_location` (31 unique combinations).

In [6]:
# Pick 5 examples covering different intents
shown, examples = set(), []
for item in fsc["train"]:
    if item["intent"] not in shown:
        examples.append(item)
        shown.add(item["intent"])
    if len(examples) == 10:
        break

for i, ex in enumerate(examples):
    raw = ex["audio"]["bytes"]
    audio, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=False)
    print(f"\n── Example {i+1} ──────────────────────────────")
    print(f"  Intent   : {ex['intent']}")
    print(f"  Action   : {action_names[ex['action']]}")
    print(f"  Object   : {object_names[ex['object']]}")
    print(f"  Location : {location_names[ex['location']]}")
    print(f"  Duration : {len(audio)/sr:.2f}s  |  SR: {sr} Hz")
    display(IPyAudio(audio, rate=sr))


── Example 1 ──────────────────────────────
  Intent   : change language_none_none
  Action   : change language
  Object   : none
  Location : none
  Duration : 1.86s  |  SR: 16000 Hz



── Example 2 ──────────────────────────────
  Intent   : activate_music_none
  Action   : activate
  Object   : music
  Location : none
  Duration : 1.39s  |  SR: 16000 Hz



── Example 3 ──────────────────────────────
  Intent   : activate_lights_none
  Action   : activate
  Object   : lights
  Location : none
  Duration : 2.14s  |  SR: 16000 Hz



── Example 4 ──────────────────────────────
  Intent   : deactivate_lights_none
  Action   : deactivate
  Object   : lights
  Location : none
  Duration : 1.95s  |  SR: 16000 Hz



── Example 5 ──────────────────────────────
  Intent   : increase_volume_none
  Action   : increase
  Object   : volume
  Location : none
  Duration : 1.76s  |  SR: 16000 Hz



── Example 6 ──────────────────────────────
  Intent   : decrease_volume_none
  Action   : decrease
  Object   : volume
  Location : none
  Duration : 2.23s  |  SR: 16000 Hz



── Example 7 ──────────────────────────────
  Intent   : increase_heat_none
  Action   : increase
  Object   : heat
  Location : none
  Duration : 1.95s  |  SR: 16000 Hz



── Example 8 ──────────────────────────────
  Intent   : decrease_heat_none
  Action   : decrease
  Object   : heat
  Location : none
  Duration : 2.14s  |  SR: 16000 Hz



── Example 9 ──────────────────────────────
  Intent   : deactivate_music_none
  Action   : deactivate
  Object   : music
  Location : none
  Duration : 1.76s  |  SR: 16000 Hz



── Example 10 ──────────────────────────────
  Intent   : activate_lamp_none
  Action   : activate
  Object   : lamp
  Location : none
  Duration : 1.86s  |  SR: 16000 Hz


## 4. Load Qwen2.5-Omni-3B (4-bit quantised — local 3060 test)

4-bit NF4 quantisation (bitsandbytes) reduces VRAM from ~6 GB → ~2 GB.  
Hidden state extraction works identically; swap this cell for full `float16` when running on RunPod.

In [7]:
from transformers import (
    Qwen2_5OmniForConditionalGeneration,
    Qwen2_5OmniProcessor,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen2.5-Omni-3B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # saves ~0.4 GB extra
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_NAME)
print("Processor loaded.")

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
)
model.eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e9
vram_used    = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f"Model loaded : {MODEL_NAME}  (4-bit NF4)")
print(f"Parameters   : {total_params:.2f}B")
print(f"VRAM used    : {vram_used:.2f} GB")

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Processor loaded.


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniToken2WavModel does not support eager attention implementation, fall back to sdpa


Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded : Qwen/Qwen2.5-Omni-3B  (4-bit NF4)
Parameters   : 3.00B
VRAM used    : 3.80 GB


## 5. Basic Inference Test

In [14]:
# Use the first example from the listening section
sample   = examples[1]
audio_np, _sr = sf.read(io.BytesIO(sample["audio"]["bytes"]), dtype="float32", always_2d=False)

TASK_PROMPT = (
    "You are classifying spoken smart-home commands. "
    "Each command has an action (e.g. activate/deactivate/increase), "
    "an object (e.g. lights/music/heat), and a location (e.g. bedroom/kitchen/none). "
    "Listen to the audio and describe the spoken command briefly."
)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": audio_np},
            {"type": "text",  "text": TASK_PROMPT},
        ],
    }
]

text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(
    text=text_input, audio=[audio_np], sampling_rate=16000, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    # return_audio=False: suppress TTS output; generate() returns plain token-ID tensor
    generated_ids = model.generate(**inputs, max_new_tokens=64, return_audio=False)

prompt_len = inputs["input_ids"].shape[1]
response   = processor.batch_decode(generated_ids[:, prompt_len:], skip_special_tokens=True)[0]

print(f"Ground-truth intent : {sample['intent']}")
print(f"Model response      : {response}")

Ground-truth intent : activate_music_none
Model response      : The spoken command is: 'Assume lights on bedroom'.


## 6. Hidden State Extraction (Sanity Check)

In [15]:
with torch.no_grad():
    # Qwen2_5OmniForConditionalGeneration.forward() is not implemented;
    # the actual LM backbone lives in model.thinker (generate() calls it the same way).
    outputs = model.thinker(**inputs, output_hidden_states=True, return_dict=True)

hidden_states = outputs.hidden_states
print(f"Hidden state layers : {len(hidden_states)}  (embedding + {len(hidden_states)-1} transformer blocks)")
print(f"Shape per layer     : {tuple(hidden_states[0].shape)}  [batch, seq_len, hidden_dim]")

mid_idx  = len(hidden_states) // 2
mid_repr = hidden_states[mid_idx].float().mean(dim=1)
print(f"\nMid-layer ({mid_idx}) mean-pooled shape : {tuple(mid_repr.shape)}")
vram_peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f"Peak VRAM (this session) : {vram_peak:.2f} GB")

Hidden state layers : 37  (embedding + 36 transformer blocks)
Shape per layer     : (1, 115, 2048)  [batch, seq_len, hidden_dim]

Mid-layer (18) mean-pooled shape : (1, 2048)
Peak VRAM (this session) : 5.16 GB
